<h1 style="background-color: gray;
           color: black;
           padding: 20px;
           text-align: center;">INFO</h1>

This script makes a tournament between PyRat players.
Players should be placed in the `players` directory, in individual directories respecting the following organization:
```
team_name
|
|_ games
|  |  # Put a simple game script here that starts a game with your program
|  |  # This will allow us to check that it works
|  |_ # Take inspiration from the visualize_XXX.ipynb game scripts with the tournament settings
|
|_ players
|  |  # Put your player here
|  |  # This directory should only contain your player file that will participate in the tournament
|  |_ # If you need functions defined in other players, please put them in utils
|
|_ utils
|  |_ # Put your additional codes here (if any)
|
|_ data
|  |_ # Put your other files here (if any)
|
|_ README_TOURNAMENT.md
```

A tournament happens in two phases:
- First, a selection allows to determine the 16 teams that will compete.
- Then a tree-based tournament is made between them.

<h1 style="background-color: gray;
           color: black;
           padding: 20px;
           text-align: center;">IMPORTS</h1>

In [ ]:
# External imports
from typing import *
from typing_extensions import *
from numbers import *
import os
import ast
import random
import importlib
import inspect
import sys
import math

# PyRat imports
from pyrat import Game, Player, PlayerSkin, GameMode

<h1 style="background-color: gray;
           color: black;
           padding: 20px;
           text-align: center;">CONSTANTS</h1>

Let's log what happens in case a player crashes, to be able to resume the tournament easily.

In [ ]:
# Path elements
SAVE_DIRECTORY = os.path.join("..", "results")
SELECTIONS_DIRECTORY = os.path.join(SAVE_DIRECTORY, "selections")
SELECTIONS_RESULTS = os.path.join(SELECTIONS_DIRECTORY, "selections.log")
TOURNAMENT_DIRECTORY = os.path.join(SAVE_DIRECTORY, "tournament")
TOURNAMENT_RESULTS = os.path.join(TOURNAMENT_DIRECTORY, "tournament.log")

The tournament will be organized in groups, before making a tree-like tournament.

In [ ]:
# Where to find the players
PLAYERS_DIRECTORY = os.path.join("..", "players")

# Max number of matches to make
MAX_MATCHES = 3

# Number of groups in the selection phase, and number of players to keep per group
# The product of both should be a power of 2
SELECTION_GROUPS = 8
KEEP_BEST = 2

Let's configure the game with a dictionary.

In [ ]:
# Customize the game elements
CONFIG = {"continue_on_error": True,
          "save_game": True,
          "mud_percentage": 20.0,
          "cell_percentage": 80.0,
          "wall_percentage": 60.0,
          "maze_width": 25,
          "mud_range": [4, 9],
          "maze_height": 20,
          "nb_cheese": 41}

<h1 style="background-color: gray;
           color: black;
           padding: 20px;
           text-align: center;">FUNCTIONS</h1>

In [ ]:
def load_players_from_team ( team_directory: str
                           ) ->              List[Player]:
    
    """
        Load the players from a team directory.
        In:
            * team_directory: the directory containing the team.
        Out:
            * players: the list of players.
    """

    # Determine the players
    team_players_directory = os.path.join(team_directory, "players")
    player_module_names = [file_name[:-3] for file_name in os.listdir(team_players_directory) if file_name.endswith(".py")]

    # Instantiate the players
    players = []
    for player_module_name in player_module_names:

        # Load module from file
        player_file_name = os.path.abspath(os.path.join(team_players_directory, player_module_name + ".py"))
        player_spec = importlib.util.spec_from_file_location(player_module_name, player_file_name)
        player_module = importlib.util.module_from_spec(player_spec)
        player_spec.loader.exec_module(player_module)
        
        # Find the player class and instantiate it
        for name, obj in inspect.getmembers(player_module, inspect.isclass):
            if issubclass(obj, Player) and obj.__module__ == player_module_name:
                player = obj()
                players.append(player)

    # Return the players
    return players

In [ ]:
def make_match ( team_1_name:       str,
                 team_2_name:       str,
                 results_file_name: str,
                 extra_game_config: Dict[str, Any],
               ) ->                 Tuple[Dict[str, Number], Dict[str, Integral], str]:
    
    """
        Make a match between two teams.
        We play up to MAX_MATCHES times.
        In:
            * team_1_name:       The name of the first team.
            * team_2_name:       The name of the second team.
            * results_file_name: The file where to save the results.
            * extra_game_config: Extra configuration for the game.
        Out:
            * times_per_team:     The time spent by each team.
            * games_won_per_team: The number of games won by each team.
            * winner:             The winner of the match.
    """

    # Print header
    print("    %s [Vs] %s" % (team_1_name, team_2_name))

    # We make up to MAX_MATCHES matches
    times_per_team = {team_1_name: 0.0, team_2_name: 0.0}
    games_won_per_team = {team_1_name: 0, team_2_name: 0}
    for match_nb in range(MAX_MATCHES):

        # Stop when it is not possible to change the winner
        if min(games_won_per_team.values()) + (MAX_MATCHES - match_nb) < max(games_won_per_team.values()):
            break

        # Check if the match has already been played
        match_already_played = False
        with open(results_file_name, "r") as results_file:
            for line in results_file:
                stats = ast.literal_eval(line)
                if isinstance(stats, dict) and "team_1" in stats and stats["team_1"] == team_1_name and stats["team_2"] == team_2_name and stats["match"] == match_nb:
                    match_already_played = True
                    break
                    
        # If needed, we play the match
        if not match_already_played:

            # Instantiate the players from the teams
            team_1 = load_players_from_team(os.path.join(PLAYERS_DIRECTORY, team_1_name))
            team_2 = load_players_from_team(os.path.join(PLAYERS_DIRECTORY, team_2_name))

            # Instantiate a game with specified arguments
            game = Game(**CONFIG, **extra_game_config)

            # Instantiate players in distinct teams
            for team_name, team, skin in zip([team_1_name, team_2_name], [team_1, team_2], [PlayerSkin.RAT, PlayerSkin.PYTHON]):
                for player in team:
                    player.skin = skin
                    game.add_player(player, team_name)

            # Start the game
            # We redirect stderr to avoid printing possible errors and have nice outputs
            old_stderr = sys.stderr
            sys.stderr = open(os.devnull, "w")
            stats = game.start()
            sys.stderr.close()
            sys.stderr = old_stderr

            # Check for errors (manual stop, timeout, etc.)
            if not stats:
                raise Exception("Game stopped")
            
            # Update stats
            stats["match"] = match_nb
            stats["team_1"] = team_1_name
            stats["team_2"] = team_2_name

            # Save the results to the selections file
            with open(results_file_name, "a") as results_file:
                results_file.write(str(stats) + "\n")
        
        # Check the winner
        scores_per_team = {team_1_name: sum([stats["players"][player_name]["score"] for player_name in stats["players"] if stats["players"][player_name]["team"] == team_1_name]),
                            team_2_name: sum([stats["players"][player_name]["score"] for player_name in stats["players"] if stats["players"][player_name]["team"] == team_2_name])}
        times_per_team[team_1_name] += sum([stats["players"][player_name]["preprocessing_duration"] or 0.0 + sum(stats["players"][player_name]["turn_durations"]) for player_name in stats["players"] if stats["players"][player_name]["team"] == team_1_name])
        times_per_team[team_2_name] += sum([stats["players"][player_name]["preprocessing_duration"] or 0.0 + sum(stats["players"][player_name]["turn_durations"]) for player_name in stats["players"] if stats["players"][player_name]["team"] == team_2_name])
        winner = team_1_name if scores_per_team[team_1_name] > scores_per_team[team_2_name] else team_2_name if scores_per_team[team_2_name] > scores_per_team[team_1_name] else "Nobody"
        if winner != "Nobody":
            games_won_per_team[winner] += 1

        # Check for errors
        players_with_errors = [player_name for player_name in stats["players"] if stats["players"][player_name]["actions"]["error"] > 0]
        errors_str = "" if len(players_with_errors) == 0 else "   # Players with errors: " + ", ".join(players_with_errors)

        # Print the game results
        print("      - Match %d: %s wins (%d - %d)%s" % (match_nb + 1, winner, max(scores_per_team[team_1_name], scores_per_team[team_2_name]), min(scores_per_team[team_1_name], scores_per_team[team_2_name]), errors_str))
    
    # Print the match results
    winner = team_1_name if games_won_per_team[team_1_name] > games_won_per_team[team_2_name] else team_2_name if games_won_per_team[team_2_name] > games_won_per_team[team_1_name] else "Nobody"
    print()
    if winner == "Nobody":
        winner = team_1_name if times_per_team[team_1_name] < times_per_team[team_2_name] else team_2_name if times_per_team[team_2_name] < times_per_team[team_1_name] else "Nobody"
        print("    Using total time to determine winner")
        print("      - %s: %.5fs" % (team_1_name, times_per_team[team_1_name]))
        print("      - %s: %.5fs" % (team_2_name, times_per_team[team_2_name]))
        print()
    print("    => Winner: %s" % (winner))
    print()

    # Return needed info
    return times_per_team, games_won_per_team, winner

In [ ]:
def print_separator ( width:      int = 80,
                      character:  str = "-",
                      line_after: bool = True
                    ) -> None:

    """
        Print a separator line.
        In:
            * width:      The width of the line.
            * character:  The character to use for the line.
            * line_after: Whether to print a line after the separator.
        Out:
            * None.
    """

    # Print the line
    print(character * width)
    if line_after:
        print()

In [ ]:
def print_title ( title:     str,
                  width:     int = 80,
                  character: str = "="
                ) ->         None:
    
    """
        Print a title.
        In:
            * title: the title to print.
            * width: the width of the line.
        Out:
            * None.
    """

    # Print the title
    print_separator(width, character, False)
    title = " " + title + " "
    print(character * math.ceil((width - len(title)) / 2) + title + character * ((width - len(title)) // 2))
    print_separator(width, character, True)

<h1 style="background-color: gray;
           color: black;
           padding: 20px;
           text-align: center;">SELECTIONS PHASE</h1>

First, split teams into balanced groups.

In [ ]:
# Parse the teams from the players directory
# No need to check if there are duplicates, as the teams are directories
team_directories = os.listdir(PLAYERS_DIRECTORY)

# If we are resuming a tournament, load the groups from the selection results (first line of the file)
if os.path.exists(SELECTIONS_RESULTS):
    with open(SELECTIONS_RESULTS, "r") as selection_results_file:
        selection_groups = ast.literal_eval(selection_results_file.readline())

# Otherwise, we create the groups
else:

    # Create the selections directory if needed
    if not os.path.exists(SELECTIONS_DIRECTORY):
        os.makedirs(SELECTIONS_DIRECTORY)

    # Shuffle the teams
    random.shuffle(team_directories)

    # Create the groups
    selection_groups = [[] for _ in range(SELECTION_GROUPS)]
    for i_team, team in enumerate(team_directories):
        selection_groups[i_team % SELECTION_GROUPS].append(team)

    # Check that all players have distinct names
    player_files_per_name = {}
    for group in selection_groups:
        for team in group:
            players = load_players_from_team(os.path.join(PLAYERS_DIRECTORY, team))
            for player in players:
                player_file_name = os.path.join(PLAYERS_DIRECTORY, team, "players", player.__class__.__name__ + ".py")
                if player.name not in player_files_per_name:
                    player_files_per_name[player.name] = []
                player_files_per_name[player.name].append(player_file_name)
    error = False
    for player_name, player_files in player_files_per_name.items():
        if len(player_files) > 1:
            print("Error: player name '%s' is not unique. Found in:" % player_name)
            for player_file in player_files:
                print("  -", player_file)
            error = True
    if error:
        raise ValueError("Player names are not unique")

    # Save the groups
    with open(SELECTIONS_RESULTS, "w") as selection_results_file:
        selection_results_file.write(str(selection_groups) + "\n")

# Print the groups
for i_group, group in enumerate(selection_groups):
    print_title("Group %d" % (i_group + 1))
    for team in group:
        print("    ", team)
    print()

Let's perform the matches!

In [ ]:
# We perform matches per group
ranking_per_group = []
for i_group, group in enumerate(selection_groups):

    # Print the group
    print_title("Group %d" % (i_group + 1))
    
    # We will attribute points depending on by how many games a team wins
    # Total time will be used to break ties
    points_per_team = {team: [0, 0.0] for team in group}

    # Make matches for all pairs of players
    for i_team_1, team_1_name in enumerate(group):
        for team_2_name in group[i_team_1+1:]:

            # Make the match
            extra_game_config = {"save_path": os.path.join(SELECTIONS_DIRECTORY, "group_ %d" % (i_group + 1), team_1_name + "___vs___" + team_2_name)}
            wins_per_team, times_per_team, winner = make_match(team_1_name, team_2_name, SELECTIONS_RESULTS, extra_game_config)
            
            # If there is a clear winner, we attribute points equal to the difference in wins + 1
            # If there is a tie, we attribute 1 point to the team with the least total time
            points = max(wins_per_team.values()) - min(wins_per_team.values()) + 1 if winner != "Nobody" else 1
            points_per_team[winner][0] += points
            points_per_team[winner][1] -= times_per_team[winner]

            # Print a separator
            if team_1_name != group[-2] or team_2_name != group[-1]:
                print_separator()

    # Print the group results
    print_separator()
    print("    Group %d results:" % (i_group + 1))
    sorted_results = sorted(group, key=lambda team: points_per_team[team], reverse=True)
    for i_team in range(len(sorted_results)):
        print("      - %s: %d pts" % (sorted_results[i_team], points_per_team[sorted_results[i_team]][0]), end="")
        if i_team > 0 and points_per_team[sorted_results[i_team]][0] == points_per_team[sorted_results[i_team - 1]][0]:
            print(" (+%.5fs)" % -(points_per_team[sorted_results[i_team]][1] - points_per_team[sorted_results[i_team - 1]][1]), end="")
        print()
    ranking_per_group.append(sorted_results)
    print()

Analyze the results to determine qualified players.

In [ ]:
# If we are resuming a tournament, load the teams from the tournament results (first line of the file)
if os.path.exists(TOURNAMENT_RESULTS):
    with open(TOURNAMENT_RESULTS, "r") as tournament_results_file:
        tournament_teams = ast.literal_eval(tournament_results_file.readline())

# Otherwise, we create the tournament teams
else:

    # Create the tournament directory if needed
    if not os.path.exists(TOURNAMENT_DIRECTORY):
        os.makedirs(TOURNAMENT_DIRECTORY)

    # Shuffle the best team from each group
    tournament_teams = [team for ranking in [ranking[:KEEP_BEST] for ranking in ranking_per_group] for team in ranking]
    random.shuffle(tournament_teams)

    # Save the teams
    with open(TOURNAMENT_RESULTS, "w") as tournament_results_file:
        tournament_results_file.write(str(tournament_teams) + "\n")

# Print the best teams
print_title("Qualified teams")
for team in tournament_teams:
    print("    ", team)

<h1 style="background-color: gray;
           color: black;
           padding: 20px;
           text-align: center;">TOURNAMENT PHASE</h1>

We make a tree-like tournament.

In [ ]:
# Start with the teams that were qualified
if os.path.exists(TOURNAMENT_RESULTS):
    with open(TOURNAMENT_RESULTS, "r") as tournament_results_file:
        tournament_teams = ast.literal_eval(tournament_results_file.readline())

# Play until there is only one team left
nb_matches = int(math.log2(len(tournament_teams)))
for round in range(nb_matches, 0, -1):

    # Print the list of teams still in the tournament
    print_title("1/%d finals" % (2 ** round) if round > 1 else "Finals")
    print("    Remaining teams:")
    for team in tournament_teams:
        print("      -", team)
    print()
    print_separator()

    # Initialize a list of winners
    winners = []

    # We make matches
    while tournament_teams:

        # Get players
        team_1_name = tournament_teams.pop(0)
        team_2_name = tournament_teams.pop(0)

        # Make the match
        extra_game_config = {"save_path": os.path.join(TOURNAMENT_DIRECTORY, team_1_name + "___vs___" + team_2_name)}
        wins_per_team, times_per_team, winner = make_match(team_1_name, team_2_name, TOURNAMENT_RESULTS, extra_game_config)

        # Add the winner to the list of winners
        winners.append(winner)

        # Print a separator
        if tournament_teams:
            print_separator()
    
    # Update the tournament teams
    tournament_teams = winners

# Print the winner
print_title("Winner")
print("    ", tournament_teams[0])